# YuE2 Studio 5.0.0 — Kaggle assets builder

**Run once with Internet ON and Accelerator None.** This prepares the official Linux runtime for the real [YuE2 Studio](https://github.com/timoncool/YuE2-Studio), plus a pinned Cloudflare Tunnel binary. It does not generate music or use a GPU.

The builder downloads the official **YuE2 Studio 5.0.0 Linux AppImage** from the v5.0.0 GitHub release and verifies it against that release's published SHA-256 manifest. It also downloads and verifies cloudflared.

## Steps
1. Run this notebook on Kaggle with Internet enabled and **Accelerator: None**.
2. Run the builder cell. The output folder is `/kaggle/working/yue2_studio_assets`.
3. Use **Save Version** and save the notebook output.
4. In the companion **YuE2 Studio 5.0.0 — Kaggle Studio** notebook, attach this notebook's output using **Add Input → Notebook Output Files**, or publish the output as a private Kaggle Dataset using the optional cell below.

This is not the former ComfyUI cover notebook. The companion notebook starts the upstream Studio 5.0.0 application backend and serves the upstream Studio React interface in a browser through Cloudflare.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import requests
import time

STUDIO_VERSION = "5.0.0"
STUDIO_TAG = "v5.0.0"
CLOUDFLARED_VERSION = "2026.10.3"
OUTPUT = Path("/kaggle/working/yue2_studio_assets")
OUTPUT.mkdir(parents=True, exist_ok=True)

GITHUB_RELEASE_API = f"https://api.github.com/repos/timoncool/YuE2-Studio/releases/tags/{STUDIO_TAG}"
CLOUDFLARED_URL = (
    f"https://github.com/cloudflare/cloudflared/releases/download/"
    f"{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"
)
CLOUDFLARED_SHA256 = "053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0"
MANIFEST_NAME = "yue2-studio-assets.json"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def fetch(url, timeout=(30, 300)):
    response = requests.get(
        url,
        headers={"User-Agent": "JJP-YuE2-Studio-Kaggle-Builder/1.0"},
        timeout=timeout,
    )
    response.raise_for_status()
    return response

def download_verified(url, destination, expected_sha256, expected_size=None):
    destination = Path(destination)
    partial = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(1, 4):
        try:
            h = hashlib.sha256()
            count = 0
            with requests.get(
                url,
                stream=True,
                headers={"User-Agent": "JJP-YuE2-Studio-Kaggle-Builder/1.0"},
                timeout=(45, 600),
            ) as response:
                response.raise_for_status()
                with open(partial, "wb") as out:
                    for block in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if not block:
                            continue
                        out.write(block)
                        h.update(block)
                        count += len(block)
            digest = h.hexdigest()
            if expected_size is not None and count != int(expected_size):
                raise RuntimeError(
                    f"Size mismatch for {destination.name}: got {count:,}, "
                    f"expected {int(expected_size):,} bytes"
                )
            if digest.lower() != expected_sha256.lower():
                raise RuntimeError(
                    f"SHA-256 mismatch for {destination.name}: {digest}"
                )
            partial.replace(destination)
            return {"name": destination.name, "size": count, "sha256": digest}
        except Exception:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise
            print(f"Download attempt {attempt} failed; retrying...")
            time.sleep(3 * attempt)

print("Checking the official YuE2 Studio release...")
release = fetch(GITHUB_RELEASE_API).json()
if release.get("tag_name") != STUDIO_TAG:
    raise RuntimeError(
        f"Expected release {STUDIO_TAG}, got {release.get('tag_name')!r}"
    )

release_assets = {item["name"]: item for item in release.get("assets", [])}
appimage_name = f"YuE2-Studio-{STUDIO_VERSION}-linux-x86_64.AppImage"
if appimage_name not in release_assets or "SHA256SUMS.txt" not in release_assets:
    raise RuntimeError("The official v5.0.0 AppImage or SHA256SUMS.txt is missing from the release.")

checksums_text = fetch(release_assets["SHA256SUMS.txt"]["browser_download_url"]).text
expected_appimage_sha = None
for line in checksums_text.splitlines():
    parts = line.strip().split()
    if len(parts) >= 2 and Path(parts[-1].lstrip("*")).name == appimage_name:
        expected_appimage_sha = parts[0].lower()
        break
if not expected_appimage_sha or not re.fullmatch(r"[0-9a-f]{64}", expected_appimage_sha):
    raise RuntimeError(f"No valid SHA-256 entry for {appimage_name} in the official release manifest.")

appimage_asset = release_assets[appimage_name]
print(f"Downloading official YuE2 Studio {STUDIO_VERSION} Linux AppImage ({appimage_asset['size']:,} bytes)...")
appimage_info = download_verified(
    appimage_asset["browser_download_url"],
    OUTPUT / appimage_name,
    expected_appimage_sha,
    appimage_asset["size"],
)
(OUTPUT / appimage_name).chmod(0o755)

print(f"Downloading cloudflared {CLOUDFLARED_VERSION}...")
cloudflared_info = download_verified(
    CLOUDFLARED_URL,
    OUTPUT / "cloudflared-linux-amd64",
    CLOUDFLARED_SHA256,
)
(OUTPUT / "cloudflared-linux-amd64").chmod(0o755)

manifest = {
    "schema": 1,
    "studio_name": "YuE2 Studio",
    "studio_version": STUDIO_VERSION,
    "studio_tag": STUDIO_TAG,
    "studio_source": "https://github.com/timoncool/YuE2-Studio",
    "studio_license": "MIT",
    "appimage_url": appimage_asset["browser_download_url"],
    "appimage": appimage_info,
    "release_checksum_manifest": release_assets["SHA256SUMS.txt"]["browser_download_url"],
    "cloudflared_version": CLOUDFLARED_VERSION,
    "cloudflared_url": CLOUDFLARED_URL,
    "cloudflared": cloudflared_info,
}
(OUTPUT / MANIFEST_NAME).write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
(OUTPUT / "README.txt").write_text(
    "YuE2 Studio 5.0.0 Kaggle runtime assets\n"
    "Official upstream Linux AppImage and checksum-pinned cloudflared binary.\n"
    "Attach this folder as an input to YuE2_Studio_5_0_0_Kaggle_Studio.ipynb.\n"
    "The Studio downloads its model components on first use from its own Models/Setup interface.\n",
    encoding="utf-8",
)
print("\nVerified assets are ready:")
for item in sorted(OUTPUT.iterdir()):
    print(f"  {item.name:52} {item.stat().st_size:>14,} bytes")
print(f"\nSave this notebook version with outputs. Asset folder: {OUTPUT}")

## Optional: publish the assets as a private Kaggle Dataset

Leave `DATASET_HANDLE` as `None` when you attach the saved notebook output directly. To reuse the same assets across notebooks, set a new private dataset slug such as `your-kaggle-username/yue2-studio-5-assets`, configure Kaggle authentication through Kaggle Secrets, and run the next cell. Do not paste API tokens into notebook code.

In [ ]:
DATASET_HANDLE = None  # Example: "your-kaggle-username/yue2-studio-5-assets"

if DATASET_HANDLE:
    import kagglehub
    kagglehub.dataset_upload(
        DATASET_HANDLE,
        str(OUTPUT),
        version_notes="Official YuE2 Studio 5.0.0 Linux runtime and checksum-verified Cloudflare tunnel binary",
    )
    print(f"Private dataset uploaded: https://www.kaggle.com/datasets/{DATASET_HANDLE}")
else:
    print("Skipped. Save the builder notebook output and attach it to the Studio runtime notebook.")